In [17]:
import pandas as pd

verification_df = pd.read_csv("nli_verification_dataset.csv")

print(verification_df.shape)
verification_df.head()

(28, 8)


,query,review_id,review_text,llm_reason,verifier_verdict,verifier_reason,review_english,human_label
0,customers complaining about damaged or defecti...,574ff7b2058b5fbb967b36ccf690c472,"Sou cliente stark há anos, mas o motivo do meu...",product was damaged and there was an issue wit...,SUPPORTED,The reviewer explicitly mentions that the prod...,"I'm a customer for years, but the reason for m...",SUPPORTED
1,customers complaining about damaged or defecti...,5b2602b06d6360ecf96806983c564cb8,"O produto veio com defeito, a cortina veio com...",product arrived with defect and damaged packaging,NOT_SUPPORTED,"The review text mentions damage and defect, bu...","The product arrived with a defect, the curtain...",SUPPORTED
2,customers complaining about damaged or defecti...,f0b8a0026b6c510825eef1286f2b27a5,O produto comprado veio incompleto. Fiz a recl...,product arrived incomplete,NOT_SUPPORTED,The review mentions the product arrived incomp...,The product I bought came incomplete. I filed ...,SUPPORTED
3,customers complaining about late delivery,574ff7b2058b5fbb967b36ccf690c472,"Sou cliente stark há anos, mas o motivo do meu...",Late delivery and damaged product,SUPPORTED,The reviewer explicitly mentions that the prod...,"I've been a customer for years, but the reason...",SUPPORTED
4,customers complaining about late delivery,f242ce44c6572a94907a446b8fda32c8,Olá!\r\nMinha queixa é a não entrega na residê...,Non-delivery and lack of communication,NOT_SUPPORTED,"The review talks about late delivery, but it d...",Hello!\nMy complaint is that there was no deli...,NOT_SUPPORTED


In [18]:
verification_df["human_label"].value_counts()

human_label
SUPPORTED        20
NOT_SUPPORTED     8
Name: count, dtype: int64

In [19]:
from transformers import pipeline

nli_pipeline = pipeline(
    "text-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [20]:
row = verification_df.iloc[27]

print("Review:")
print(row["review_text"])

print("\nClaim:")
print(row["llm_reason"])

Review:
O produto veio com dimensões erradas e avariado.

Claim:
O produto veio com dimensões erradas e avariado.


In [21]:
result = nli_pipeline(
    row["review_text"],
    text_pair=row["llm_reason"]
)

result

[{'label': 'entailment', 'score': 0.9989656209945679}]

In [22]:
def nli_verify(review_text, claim):
    result = nli_pipeline(
        review_text,
        text_pair=claim
    )[0]

    return result["label"], result["score"]

In [27]:
row = verification_df.iloc[25]

print("Review:")
print(row["review_text"])

print("\nClaim:")
print(row["llm_reason"])

nli_verify(
    row["review_text"],
    row["llm_reason"]
)

Review:
Insatisfação com a descrição do produto.

Claim:
Insatisfação com a descrição do produto.


('entailment', 0.9981719255447388)

In [32]:
human_labels = [
    "SUPPORTED",       # 0
    "SUPPORTED",       # 1
    "SUPPORTED",       # 2
    "SUPPORTED",       # 3
    "NOT_SUPPORTED",   # 4
    "SUPPORTED",       # 5
    "SUPPORTED",       # 6
    "SUPPORTED",       # 7
    "SUPPORTED",       # 8
    "SUPPORTED",       # 9
    "SUPPORTED",       # 10  <-- corrected
    "SUPPORTED",       # 11
    "SUPPORTED",       # 12
    "SUPPORTED",       # 13
    "SUPPORTED",       # 14
    "SUPPORTED",       # 15
    "SUPPORTED",       # 16
    "NOT_SUPPORTED",   # 17
    "NOT_SUPPORTED",   # 18
    "NOT_SUPPORTED",   # 19
    "SUPPORTED",       # 20
    "NOT_SUPPORTED",   # 21
    "SUPPORTED",       # 22
    "SUPPORTED",       # 23
    "SUPPORTED",       # 24
    "SUPPORTED",       # 25
    "SUPPORTED",       # 26
    "SUPPORTED"        # 27
]

verification_df["human_label"] = human_labels

verification_df["human_label"].value_counts()

human_label
SUPPORTED        23
NOT_SUPPORTED     5
Name: count, dtype: int64

In [35]:
nli_results = []

for _, row in verification_df.iterrows():
    label, score = nli_verify(
        row["review_text"],
        row["llm_reason"]
    )

    nli_results.append({
        "nli_label": label,
        "nli_score": score
    })

verification_df["nli_label"] = [
    x["nli_label"] for x in nli_results
]

verification_df["nli_score"] = [
    x["nli_score"] for x in nli_results
]

In [37]:
nli_results

[{'nli_label': 'neutral', 'nli_score': 0.9782440662384033},
 {'nli_label': 'entailment', 'nli_score': 0.9986504912376404},
 {'nli_label': 'entailment', 'nli_score': 0.9966447353363037},
 {'nli_label': 'neutral', 'nli_score': 0.9967988729476929},
 {'nli_label': 'entailment', 'nli_score': 0.9465581178665161},
 {'nli_label': 'entailment', 'nli_score': 0.998173713684082},
 {'nli_label': 'entailment', 'nli_score': 0.976100742816925},
 {'nli_label': 'entailment', 'nli_score': 0.9966660141944885},
 {'nli_label': 'entailment', 'nli_score': 0.9987884163856506},
 {'nli_label': 'entailment', 'nli_score': 0.9973898530006409},
 {'nli_label': 'entailment', 'nli_score': 0.9945999383926392},
 {'nli_label': 'entailment', 'nli_score': 0.9817245006561279},
 {'nli_label': 'entailment', 'nli_score': 0.9874744415283203},
 {'nli_label': 'entailment', 'nli_score': 0.994657576084137},
 {'nli_label': 'entailment', 'nli_score': 0.9859091639518738},
 {'nli_label': 'entailment', 'nli_score': 0.972018837928772},
 {

In [38]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Convert NLI output to our binary task
verification_df["nli_binary"] = verification_df["nli_label"].apply(
    lambda x: "SUPPORTED"
    if x == "entailment"
    else "NOT_SUPPORTED"
)

y_true = verification_df["human_label"]
y_pred = verification_df["nli_binary"]

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    pos_label="SUPPORTED"
)

recall = recall_score(
    y_true,
    y_pred,
    pos_label="SUPPORTED"
)

f1 = f1_score(
    y_true,
    y_pred,
    pos_label="SUPPORTED"
)

print(f"Accuracy : {accuracy:.2%}")
print(f"Precision: {precision:.2%}")
print(f"Recall   : {recall:.2%}")
print(f"F1       : {f1:.2%}")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_true,
        y_pred,
        labels=["SUPPORTED", "NOT_SUPPORTED"]
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=["SUPPORTED", "NOT_SUPPORTED"]
    )
)

Accuracy : 82.14%
Precision: 90.91%
Recall   : 86.96%
F1       : 88.89%

Confusion Matrix:
[[20  3]
 [ 2  3]]

Classification Report:
               precision    recall  f1-score   support

    SUPPORTED       0.91      0.87      0.89        23
NOT_SUPPORTED       0.50      0.60      0.55         5

     accuracy                           0.82        28
    macro avg       0.70      0.73      0.72        28
 weighted avg       0.84      0.82      0.83        28



### testing all 3 possibilities,

entailment: 0.55
neutral:    0.43
contradiction: 0.02

We probably shouldn't confidently accept it.
So let's retrieve all three NLI probabilities.

In [39]:
def nli_verify_all(review_text, claim):

    result = nli_pipeline(
        review_text,
        text_pair=claim,
        top_k=None
    )

    return {
        item["label"]: item["score"]
        for item in result
    }

In [40]:
test = nli_verify_all(
    verification_df.iloc[0]["review_text"],
    verification_df.iloc[0]["llm_reason"]
)

test

{'neutral': 0.9782440662384033,
 'entailment': 0.021643908694386482,
 'contradiction': 0.00011208927753614262}

In [41]:
nli_probabilities = []

for _, row in verification_df.iterrows():

    probs = nli_verify_all(
        row["review_text"],
        row["llm_reason"]
    )

    nli_probabilities.append(probs)

In [42]:
verification_df["entailment_score"] = [
    x["entailment"]
    for x in nli_probabilities
]

verification_df["neutral_score"] = [
    x["neutral"]
    for x in nli_probabilities
]

verification_df["contradiction_score"] = [
    x["contradiction"]
    for x in nli_probabilities
]

In [43]:
verification_df[
    [
        "human_label",
        "nli_label",
        "entailment_score",
        "neutral_score",
        "contradiction_score"
    ]
]

,human_label,nli_label,entailment_score,neutral_score,contradiction_score
0,SUPPORTED,neutral,0.021644,0.978244,0.000112
1,SUPPORTED,entailment,0.998650,0.001246,0.000104
2,SUPPORTED,entailment,0.996645,0.003292,0.000063
3,SUPPORTED,neutral,0.003036,0.996799,0.000165
4,NOT_SUPPORTED,entailment,0.946558,0.051012,0.002430
5,SUPPORTED,entailment,0.998174,0.001671,0.000155
6,SUPPORTED,entailment,0.976101,0.023696,0.000203
7,SUPPORTED,entailment,0.996666,0.003208,0.000126
8,SUPPORTED,entailment,0.998788,0.001108,0.000103
9,SUPPORTED,entailment,0.997390,0.002477,0.000133


In [44]:
verification_df[
    verification_df["human_label"] == "NOT_SUPPORTED"
][
    [
        "review_text",
        "llm_reason",
        "entailment_score",
        "neutral_score",
        "contradiction_score"
    ]
]

,review_text,llm_reason,entailment_score,neutral_score,contradiction_score
4,Olá!\r\nMinha queixa é a não entrega na residê...,Non-delivery and lack of communication,0.946558,0.051012,0.002430
17,"Sou cliente stark há anos, mas o motivo do meu...",Wrong product received,0.000588,0.999084,0.000328
18,eu recebi o produto com defeito quero devolver...,Wrong product received,0.991076,0.008600,0.000325
19,Atraso na entrega sem dar satisfação ao client...,Wrong product received,0.461268,0.039008,0.499724
21,"Recebi a compra pela metade, comprei 03 kits e...",péssima compra,0.388306,0.610891,0.000803


In [45]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

threshold_results = []

thresholds = np.arange(0.0, 1.01, 0.05)

for threshold in thresholds:

    predictions = np.where(
        verification_df["entailment_score"] >= threshold,
        "SUPPORTED",
        "NOT_SUPPORTED"
    )

    y_true = verification_df["human_label"]

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(
            y_true,
            predictions,
            pos_label="SUPPORTED",
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            predictions,
            pos_label="SUPPORTED",
            zero_division=0
        ),
        "f1": f1_score(
            y_true,
            predictions,
            pos_label="SUPPORTED",
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,threshold,accuracy,precision,recall,f1
0,0.00,0.821429,0.821429,1.000000,0.901961
1,0.05,0.785714,0.840000,0.913043,0.875000
2,0.10,0.750000,0.833333,0.869565,0.851064
3,0.15,0.750000,0.833333,0.869565,0.851064
4,0.20,0.750000,0.833333,0.869565,0.851064
5,0.25,0.750000,0.833333,0.869565,0.851064
6,0.30,0.750000,0.833333,0.869565,0.851064
7,0.35,0.750000,0.833333,0.869565,0.851064
8,0.40,0.785714,0.869565,0.869565,0.869565
9,0.45,0.785714,0.869565,0.869565,0.869565


In [46]:
threshold_df.sort_values(
    "f1",
    ascending=False
)

,threshold,accuracy,precision,recall,f1
19,0.95,0.857143,0.952381,0.869565,0.909091
0,0.00,0.821429,0.821429,1.000000,0.901961
11,0.55,0.821429,0.909091,0.869565,0.888889
18,0.90,0.821429,0.909091,0.869565,0.888889
17,0.85,0.821429,0.909091,0.869565,0.888889
16,0.80,0.821429,0.909091,0.869565,0.888889
15,0.75,0.821429,0.909091,0.869565,0.888889
14,0.70,0.821429,0.909091,0.869565,0.888889
13,0.65,0.821429,0.909091,0.869565,0.888889
12,0.60,0.821429,0.909091,0.869565,0.888889


In [47]:
threshold_df[
    threshold_df["threshold"].isin([0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95])
]

,threshold,accuracy,precision,recall,f1
10,0.5,0.821429,0.909091,0.869565,0.888889
16,0.8,0.821429,0.909091,0.869565,0.888889
18,0.9,0.821429,0.909091,0.869565,0.888889


In [48]:
threshold = 0.95

verification_df["threshold_prediction"] = np.where(
    verification_df["entailment_score"] >= threshold,
    "SUPPORTED",
    "NOT_SUPPORTED"
)

In [49]:
errors = verification_df[
    verification_df["threshold_prediction"] != verification_df["human_label"]
][
    [
        "query",
        "review_text",
        "llm_reason",
        "human_label",
        "entailment_score",
        "threshold_prediction"
    ]
]

errors

,query,review_text,llm_reason,human_label,entailment_score,threshold_prediction
0,customers complaining about damaged or defecti...,"Sou cliente stark há anos, mas o motivo do meu...",product was damaged and there was an issue wit...,SUPPORTED,0.021644,NOT_SUPPORTED
3,customers complaining about late delivery,"Sou cliente stark há anos, mas o motivo do meu...",Late delivery and damaged product,SUPPORTED,0.003036,NOT_SUPPORTED
16,customers asking for refunds or replacements,eu recebi o produto com defeito quero devolver...,product with defect and unresponsive customer ...,SUPPORTED,0.064814,NOT_SUPPORTED
18,customers receiving the wrong product,eu recebi o produto com defeito quero devolver...,Wrong product received,NOT_SUPPORTED,0.991076,SUPPORTED


In [50]:
verification_df.to_csv(
    "nli_verification_results.csv",
    index=False
)

NLI_THRESHOLD = 0.95